# Moshabeh — Development and external validation of an interpretable cognitive-screening model

**Locked clinical question:** predict *6CIT-defined screen-positive cognitive impairment* (`6CIT ≥ 8`) using harmonized baseline predictors.

- Development cohort: 2,000 participants (945 MCI, 1,055 non-MCI)
- External cohort: 1,165 participants (614 MCI, 551 non-MCI)
- Missingness exclusion: predictor missingness >25%
- Correlation exclusion: absolute Spearman rho >=0.80
- Small baseline: age + sex + education
- Article-based benchmarks: TabPFN, full logistic regression, SVM, RF, GBM, XGBoost, CatBoost, LightGBM
- Validation: nested stratified 5-fold outer / 3-fold inner CV, followed by locked external validation
- Interpretability: SHAP, permutation importance, subgroup analyses (sex and age), and education sensitivity analysis

The external outcome is not accessed until the model family, preprocessing, predictor list, hyperparameters and screening threshold have been locked. `Code` and `id` are never included in the analytic dataframe or outputs.

The reporting layout follows the example paper: baseline table; multi-model ROC/PR/calibration/DCA panels; cross-model feature-importance comparison; SHAP figures; stability, subgroup and sensitivity analyses in the Supplementary Material.


In [ ]:
# 0. Reproducible environment, TabPFN-3, and output structure
import os, sys, glob, json, random, platform, warnings, importlib, subprocess
from pathlib import Path

# The licensed TabPFN-3 checkpoint requires an enabled Kaggle Secret named
# TABPFN_TOKEN. Check it before installing packages or fitting other models.
os.environ["TABPFN_NO_BROWSER"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    _tabpfn_token = UserSecretsClient().get_secret("TABPFN_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Kaggle Secret TABPFN_TOKEN is unavailable. Enable it for this notebook "
        "and store a current Prior Labs API key in its VALUE."
    ) from exc
if not _tabpfn_token or not _tabpfn_token.strip():
    raise RuntimeError(
        "Kaggle Secret TABPFN_TOKEN is empty. Store a current Prior Labs API key "
        "in the secret VALUE and accept the TabPFN-3 model license."
    )
os.environ["TABPFN_TOKEN"] = _tabpfn_token.strip()
print("Prior Labs token loaded from enabled Kaggle Secret: TABPFN_TOKEN")

# TabPFN package 8.x uses TabPFN-3 as the default checkpoint.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "tabpfn==8.5.0"],
    check=True,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
warnings.filterwarnings('ignore')

ROOT = Path('/kaggle/working/moshabeh')
FIG_MAIN = ROOT / 'figures_main'
FIG_SUPP = ROOT / 'figures_supplementary'
TABLES = ROOT / 'tables'
ARTIFACTS = ROOT / 'artifacts'
for p in [ROOT, FIG_MAIN, FIG_SUPP, TABLES, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper')
PALETTE = sns.color_palette('colorblind', 10)

def find_input(filename):
    hits = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    assert len(hits) == 1, f'Expected one {filename}; found {hits}'
    return hits[0]

versions = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'tabpfn_target': '3 (package 8.5.0)',
}
print('Output directory:', ROOT)
print(json.dumps(versions, indent=2))

## 1. Harmonization rules and locked predictor set

The 43 agreed candidate predictors are harmonized without using the external outcome. Four predictors are removed for >25% missingness in the transport setting, and three clinically redundant predictors are removed for high correlation. The resulting 36-predictor set is locked before modeling.


In [ ]:
# 1. Cohort-specific cleaning and harmonization
NUMERIC_43 = [
    'age','BMI','waist_circumference_cm','waist_hip_ratio','hip_circumference_cm',
    'PLT','HCT','Triglycerides','rbc','wbc','hemoglobin','total_cholesterol',
    'hdl_cholesterol','ldl_cholesterol','uric_acid','ast','alt','vitamin_d',
    'whisper_test_left','whisper_test_right','visual_acuity_both','max_handgrip',
    'TOTAL_PFAT','ANDROID_PFAT','GYNOID_PFAT','Albumin'
]
CATEGORICAL_43 = [
    'sex','employment_status','housing_status','marital_status','household_income',
    'smoking_status','sleep_quality','adl','iadl','osteoporosis_status',
    'depression_status','diabetes','hypertension','history_heart_failure',
    'history_stroke','history_myocardial_infarction','EDU'
]
CANDIDATES_43 = NUMERIC_43 + CATEGORICAL_43
assert len(CANDIDATES_43) == 43

DROP_MISSING_GT25 = ['vitamin_d','TOTAL_PFAT','ANDROID_PFAT','GYNOID_PFAT']
DROP_HIGH_CORR = ['waist_circumference_cm','hip_circumference_cm','total_cholesterol']
FEATURES_36 = [f for f in CANDIDATES_43 if f not in DROP_MISSING_GT25 + DROP_HIGH_CORR]
NUMERIC_36 = [f for f in NUMERIC_43 if f in FEATURES_36]
CATEGORICAL_36 = [f for f in CATEGORICAL_43 if f in FEATURES_36]
assert len(FEATURES_36) == 36

MISSING_CODES = {'99','',' ','nan','na','.','none'}

def clean_string(series):
    z = series.astype('string').str.strip()
    return z.where(~z.str.lower().isin(MISSING_CODES), pd.NA).str.lower()

def clean_development(raw):
    d = pd.DataFrame(index=raw.index)
    right = pd.to_numeric(raw['max_handgrip_right'].astype(str).str.strip(), errors='coerce').replace(0, np.nan)
    left = pd.to_numeric(raw['max_handgrip_left'].astype(str).str.strip(), errors='coerce').replace(0, np.nan)
    d['max_handgrip'] = np.fmax(right, left)
    for col in NUMERIC_43:
        if col != 'max_handgrip':
            d[col] = pd.to_numeric(raw[col].astype(str).str.strip(), errors='coerce')
    for col in CATEGORICAL_43:
        d[col] = clean_string(raw[col])
    d['smoking_status'] = d['smoking_status'].map(
        lambda v: pd.NA if pd.isna(v) else ('no' if v == 'no' else 'yes'))
    d['housing_status'] = d['housing_status'].map(
        lambda v: pd.NA if pd.isna(v) else ('indep_spouse' if 'spous' in v else 'with_others'))
    d['EDU'] = d['EDU'].where(d['EDU'].isin({'illiterate&primary','secondary','academic'}), pd.NA)
    d['MCI'] = clean_string(raw['MCI']).map({'yes':1, 'no':0})
    d = d.dropna(subset=['MCI']).reset_index(drop=True)
    d['MCI'] = d['MCI'].astype(int)
    for col in CATEGORICAL_43:
        d[col] = d[col].astype(object).where(d[col].notna(), np.nan)
    return d

EXT_NUMERIC_MAP = {
    'age':'age','BMI':'BMI','waist_circumference_cm':'Waistcm','waist_hip_ratio':'WHR',
    'hip_circumference_cm':'Hipcm','PLT':'plt','HCT':'hct','Triglycerides':'tg',
    'rbc':'rbc','wbc':'wbc','hemoglobin':'hgb','total_cholesterol':'chol',
    'hdl_cholesterol':'hdl','ldl_cholesterol':'ldl','uric_acid':'UricAcid',
    'ast':'ast','alt':'alt','vitamin_d':'VitD','whisper_test_left':'WSPR_LEFT',
    'whisper_test_right':'WSPR_RIGHT','visual_acuity_both':'VISACU_BOTH',
    'max_handgrip':'max_Hndgrp','TOTAL_PFAT':'TOTAL_PFAT','ANDROID_PFAT':'ANDROID_PFAT',
    'GYNOID_PFAT':'GYNOID_PFAT','Albumin':'Albumin'
}
EXT_CATEGORICAL_MAP = {
    'sex':'sex','employment_status':'Job','housing_status':'homep','marital_status':'marital_new',
    'household_income':'wealth_index','smoking_status':'Current_Smoker',
    'sleep_quality':'Global_PSQI_Binary','adl':'Disable_ADL','iadl':'Disable_IADL',
    'osteoporosis_status':'osteoporosis_final2','depression_status':'depression',
    'diabetes':'dm','hypertension':'htn','history_heart_failure':'CHF01',
    'history_stroke':'Stroke01','history_myocardial_infarction':'MI01','EDU':'Education'
}

def clean_external_predictors(raw):
    d = pd.DataFrame(index=raw.index)
    for canonical, source in EXT_NUMERIC_MAP.items():
        d[canonical] = pd.to_numeric(raw[source].astype(str).str.strip(), errors='coerce')
    for canonical, source in EXT_CATEGORICAL_MAP.items():
        d[canonical] = clean_string(raw[source])
    d['household_income'] = d['household_income'].replace({
        'poorest':'low','poorer':'low','middle':'moderate','richer':'high','richest':'high'})
    d['marital_status'] = d['marital_status'].replace({'m':'married','s':'single'})
    d['housing_status'] = d['housing_status'].map(
        lambda v: pd.NA if pd.isna(v) else ('indep_spouse' if 'independ' in v else 'with_others'))
    d['EDU'] = d['EDU'].map(lambda v: pd.NA if pd.isna(v) else (
        'academic' if 'academic' in v else (
        'secondary' if 'second' in v else (
        'illiterate&primary' if ('illit' in v or 'primary' in v) else pd.NA))))
    d['smoking_status'] = d['smoking_status'].map(
        lambda v: pd.NA if pd.isna(v) else ('no' if v in {'no','0'} else 'yes'))
    for col in CATEGORICAL_43:
        d[col] = d[col].astype(object).where(d[col].notna(), np.nan)
    return d.reset_index(drop=True)

def clean_external_outcome(raw):
    y = clean_string(raw['MCI']).replace({'yes':1,'no':0,'1':1,'0':0})
    assert y.notna().all(), 'External outcome contains unmapped or missing values.'
    return y.astype(int).to_numpy()


## 2. Stage 1 — data quality, transportability and feature lock

This stage reads the external *predictors* only. External MCI labels remain sealed. The outputs provide the data audit, missingness table, correlation audit, and the locked 36-predictor manifest.


In [ ]:
# 2. Load data and perform the locked data-quality audit
dev_path = find_input('Development_corrected_with_Albumin.xlsx')
ext_path = find_input('External_with_Albumin.xlsx')
dev_raw = pd.read_excel(dev_path, sheet_name='Data')
ext_raw = pd.read_excel(ext_path, sheet_name='Data')

dev = clean_development(dev_raw)
ext_X = clean_external_predictors(ext_raw)
y_dev = dev['MCI'].to_numpy()

assert len(dev) == 2000
assert int(y_dev.sum()) == 945
assert len(ext_X) == 1165
assert 'Code' not in ext_X.columns and 'id' not in ext_X.columns
assert set(FEATURES_36).issubset(dev.columns) and set(FEATURES_36).issubset(ext_X.columns)

missingness = pd.DataFrame({
    'feature': CANDIDATES_43,
    'development_missing_pct': [100*dev[f].isna().mean() for f in CANDIDATES_43],
    'external_missing_pct': [100*ext_X[f].isna().mean() for f in CANDIDATES_43],
})
missingness['removed_gt25'] = missingness['feature'].isin(DROP_MISSING_GT25)
missingness['removed_high_corr'] = missingness['feature'].isin(DROP_HIGH_CORR)
missingness['retained_final'] = missingness['feature'].isin(FEATURES_36)
missingness.to_csv(TABLES/'S7_feature_availability_and_exclusions.csv', index=False)

corr = dev[NUMERIC_43].corr(method='spearman')
high_pairs = []
for i, a in enumerate(NUMERIC_43):
    for b in NUMERIC_43[i+1:]:
        rho = corr.loc[a,b]
        if pd.notna(rho) and abs(rho) >= 0.80:
            high_pairs.append({'feature_1':a,'feature_2':b,'spearman_rho':rho})
high_pairs = pd.DataFrame(
    high_pairs, columns=['feature_1','feature_2','spearman_rho']
).sort_values('spearman_rho', key=lambda s:s.abs(), ascending=False)
high_pairs.to_csv(TABLES/'correlation_pairs_abs_rho_ge_0.80.csv', index=False)

lock = {
    'stage':'FEATURE_LOCK',
    'external_outcome_accessed':False,
    'development_n':len(dev),
    'development_events':int(y_dev.sum()),
    'external_predictor_rows':len(ext_X),
    'candidate_predictors':43,
    'missingness_cutoff_pct':25,
    'dropped_missing_gt25':DROP_MISSING_GT25,
    'correlation_cutoff_abs_spearman':0.80,
    'dropped_high_correlation':DROP_HIGH_CORR,
    'final_predictors':FEATURES_36,
    'final_predictor_count':len(FEATURES_36),
    'target':'6CIT-defined screen-positive cognitive impairment (6CIT ≥ 8)',
    'seed':SEED,
}
with open(ARTIFACTS/'stage1_feature_lock.json','w',encoding='utf-8') as f:
    json.dump(lock,f,indent=2)

print(f'Development: n={len(dev)}, MCI={y_dev.sum()} ({y_dev.mean():.1%})')
print(f'External predictors: n={len(ext_X)}; outcome still sealed')
print('43 candidates -> 39 after >25% missingness -> 36 after correlation pruning')
print('Final predictors:', FEATURES_36)
display(missingness.sort_values('external_missing_pct', ascending=False).head(12))
display(high_pairs)


In [ ]:
# 3. Stage-1 publication figures: cohort overview, missingness, correlation
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel A: development outcome distribution
counts = pd.Series(y_dev).map({0:'Non-MCI',1:'MCI'}).value_counts().reindex(['Non-MCI','MCI'])
axes[0].bar(counts.index, counts.values, color=[PALETTE[0],PALETTE[1]])
for i,v in enumerate(counts.values):
    axes[0].text(i,v+20,f'{v}\n({v/len(dev):.1%})',ha='center')
axes[0].set(title='A. Development outcome', ylabel='Participants')

# Panel B: missingness comparison
plot_miss = missingness.sort_values('external_missing_pct', ascending=False).head(15).iloc[::-1]
ypos = np.arange(len(plot_miss))
axes[1].barh(ypos-0.18, plot_miss['development_missing_pct'], height=.34, label='Development', color=PALETTE[0])
axes[1].barh(ypos+0.18, plot_miss['external_missing_pct'], height=.34, label='External', color=PALETTE[1])
axes[1].axvline(25,color='black',ls='--',lw=1,label='25% cutoff')
axes[1].set_yticks(ypos,plot_miss['feature'])
axes[1].set(xlabel='Missing (%)',title='B. Predictor missingness')
axes[1].legend(frameon=False,fontsize=8)

# Panel C: final numeric correlation matrix
final_corr = dev[NUMERIC_36].corr(method='spearman')
sns.heatmap(final_corr, cmap='vlag', center=0, vmin=-1, vmax=1, ax=axes[2], cbar_kws={'label':'Spearman rho'})
axes[2].set_title('C. Retained numeric predictors')
axes[2].tick_params(axis='x',rotation=90,labelsize=7)
axes[2].tick_params(axis='y',labelsize=7)

fig.suptitle('Figure 1. Cohort and predictor quality audit',fontsize=14)
fig.tight_layout()
fig.savefig(FIG_MAIN/'Figure1_data_quality.png',dpi=300,bbox_inches='tight')
fig.savefig(FIG_MAIN/'Figure1_data_quality.pdf',bbox_inches='tight')
plt.show()

# Supplementary full missingness heatmap
fig, ax = plt.subplots(figsize=(10,10))
miss_mat = pd.DataFrame({
    'Development':dev[CANDIDATES_43].isna().mean()*100,
    'External':ext_X[CANDIDATES_43].isna().mean()*100,
}).sort_values('External',ascending=True)
sns.heatmap(miss_mat,annot=True,fmt='.1f',cmap='viridis_r',cbar_kws={'label':'Missing (%)'},ax=ax)
ax.axhline(len(miss_mat)-4,color='red',lw=1)
ax.set_title('Supplementary Figure S1. Feature availability across cohorts')
fig.tight_layout()
fig.savefig(FIG_SUPP/'FigureS1_missingness.png',dpi=300,bbox_inches='tight')
fig.savefig(FIG_SUPP/'FigureS1_missingness.pdf',bbox_inches='tight')
plt.show()


## 3. Stage 2 — leakage-safe preprocessing and nested model comparison

Every imputer, missingness indicator, encoder, scaler and hyperparameter search is fitted inside the training fold. No SMOTE or class weighting is used because the development outcome is nearly balanced. The small demographic logistic model is kept separate from the full elastic-net logistic comparator.


In [ ]:
# 4. Modeling helpers
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, SplineTransformer
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, roc_curve,
    precision_recall_curve, confusion_matrix, accuracy_score, recall_score,
    precision_score, f1_score, matthews_corrcoef
)

OUTER_SPLITS = 5
INNER_SPLITS = 3
TARGET_SENSITIVITY = 0.80
DEMO_FEATURES = ['age','sex','EDU']

def make_preprocessor(features, ordinal=False, demographic=False):
    num = [f for f in features if f in NUMERIC_36]
    cat = [f for f in features if f in CATEGORICAL_36]
    if demographic:
        age_pipe = Pipeline([
            ('impute',SimpleImputer(strategy='median')),
            ('spline',SplineTransformer(n_knots=3,degree=3,include_bias=False)),
            ('scale',RobustScaler())
        ])
        cat_pipe = Pipeline([
            ('impute',SimpleImputer(strategy='most_frequent')),
            ('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))
        ])
        return ColumnTransformer([('age',age_pipe,['age']),('cat',cat_pipe,cat)],sparse_threshold=0)
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1) if ordinal else OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    num_pipe = Pipeline([
        ('impute',IterativeImputer(random_state=SEED,max_iter=10,initial_strategy='median',add_indicator=True,skip_complete=True)),
        ('scale',RobustScaler())
    ])
    cat_pipe = Pipeline([
        ('impute',SimpleImputer(strategy='most_frequent',add_indicator=True)),
        ('encode',encoder)
    ])
    return ColumnTransformer([('num',num_pipe,num),('cat',cat_pipe,cat)],sparse_threshold=0)

def calibration_intercept_slope(y, p):
    p = np.clip(np.asarray(p),1e-6,1-1e-6)
    logit = np.log(p/(1-p)).reshape(-1,1)
    lr = LogisticRegression(C=1e6,solver='lbfgs',max_iter=2000).fit(logit,y)
    return float(lr.intercept_[0]), float(lr.coef_[0,0])

def threshold_for_sensitivity(y, p, target=TARGET_SENSITIVITY):
    candidates = np.sort(np.unique(p))[::-1]
    valid = [t for t in candidates if recall_score(y,(p>=t).astype(int)) >= target]
    return float(max(valid)) if valid else 0.5

def metric_row(y, p, threshold=0.5):
    q = (np.asarray(p)>=threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,q,labels=[0,1]).ravel()
    ci,cs = calibration_intercept_slope(y,p)
    return {
        'AUROC':roc_auc_score(y,p),'AUPRC':average_precision_score(y,p),
        'Brier':brier_score_loss(y,p),'calibration_intercept':ci,'calibration_slope':cs,
        'threshold':threshold,'accuracy':accuracy_score(y,q),'sensitivity':recall_score(y,q),
        'specificity':tn/(tn+fp),'PPV':precision_score(y,q,zero_division=0),
        'NPV':tn/(tn+fn) if (tn+fn)>0 else np.nan,'F1':f1_score(y,q,zero_division=0),
        'MCC':matthews_corrcoef(y,q),'TN':tn,'FP':fp,'FN':fn,'TP':tp,
    }

def bootstrap_ci(y,p,metric,n_boot=1000,seed=SEED):
    rng=np.random.default_rng(seed); vals=[]; n=len(y)
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        if len(np.unique(y[idx]))<2: continue
        vals.append(metric(y[idx],p[idx]))
    return np.quantile(vals,[.025,.975])

def net_benefit(y,p,thresholds):
    n=len(y); out=[]
    for t in thresholds:
        q=p>=t; tp=((q)&(y==1)).sum(); fp=((q)&(y==0)).sum()
        out.append(tp/n-fp/n*(t/(1-t)))
    return np.asarray(out)


In [ ]:
# 5. Article-based model set (optional libraries are used only when already available)
MODEL_SPECS = {}

MODEL_SPECS['DemographicLR'] = {
    'features':DEMO_FEATURES,
    'pipeline':Pipeline([('prep',make_preprocessor(DEMO_FEATURES,demographic=True)),
                         ('model',LogisticRegression(max_iter=5000,random_state=SEED))]),
    'grid':{'model__C':[0.1,1,10]}
}
MODEL_SPECS['ElasticNet'] = {
    'features':FEATURES_36,
    'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
                         ('model',LogisticRegression(penalty='elasticnet',solver='saga',max_iter=6000,random_state=SEED))]),
    'grid':{'model__C':[0.1,1,10],'model__l1_ratio':[0,0.5,1]}
}
MODEL_SPECS['SVM'] = {
    'features':FEATURES_36,
    'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
                         ('model',SVC(probability=True,random_state=SEED))]),
    'grid':{'model__C':[0.5,1,2],'model__gamma':['scale',0.01]}
}
MODEL_SPECS['RF'] = {
    'features':FEATURES_36,
    'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
                         ('model',RandomForestClassifier(n_estimators=400,random_state=SEED,n_jobs=1))]),
    'grid':{'model__max_depth':[None,8],'model__min_samples_leaf':[1,5]}
}
MODEL_SPECS['GBM'] = {
    'features':FEATURES_36,
    'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
                         ('model',GradientBoostingClassifier(random_state=SEED))]),
    'grid':{'model__n_estimators':[150,300],'model__learning_rate':[0.03,0.08],'model__max_depth':[1,2]}
}

optional_status = {}
try:
    from xgboost import XGBClassifier
    MODEL_SPECS['XGBoost'] = {
        'features':FEATURES_36,
        'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
            ('model',XGBClassifier(n_estimators=300,eval_metric='logloss',random_state=SEED,n_jobs=1))]),
        'grid':{'model__max_depth':[2,4],'model__learning_rate':[0.03,0.08],'model__subsample':[0.8,1.0]}}
    optional_status['XGBoost']='available'
except Exception as e: optional_status['XGBoost']=f'unavailable: {e}'

try:
    from catboost import CatBoostClassifier
    MODEL_SPECS['CatBoost'] = {
        'features':FEATURES_36,
        'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
            ('model',CatBoostClassifier(verbose=0,allow_writing_files=False,random_seed=SEED,thread_count=1))]),
        'grid':{'model__iterations':[300,600],'model__depth':[4,6],'model__learning_rate':[0.03,0.08]}}
    optional_status['CatBoost']='available'
except Exception as e: optional_status['CatBoost']=f'unavailable: {e}'

try:
    from lightgbm import LGBMClassifier
    MODEL_SPECS['LightGBM'] = {
        'features':FEATURES_36,
        'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36)),
            ('model',LGBMClassifier(n_estimators=400,random_state=SEED,n_jobs=1,verbosity=-1))]),
        'grid':{'model__num_leaves':[7,15],'model__learning_rate':[0.03,0.08],'model__min_child_samples':[10,30]}}
    optional_status['LightGBM']='available'
except Exception as e: optional_status['LightGBM']=f'unavailable: {e}'

try:
    import torch
    from tabpfn import TabPFNClassifier
    tp_device='cuda' if torch.cuda.is_available() else 'cpu'
    MODEL_SPECS['TabPFN'] = {
        'features':FEATURES_36,
        'pipeline':Pipeline([('prep',make_preprocessor(FEATURES_36,ordinal=True)),
            ('model',TabPFNClassifier(device=tp_device,ignore_pretraining_limits=True))]),
        'grid':{}}
    optional_status['TabPFN']=f'available ({tp_device})'
except Exception as e: optional_status['TabPFN']=f'unavailable: {e}'

print('Models:',list(MODEL_SPECS))
print(json.dumps(optional_status,indent=2))
if 'TabPFN' not in MODEL_SPECS:
    print('TabPFN is not installed in this Kaggle image. All other stages remain runnable; do not install packages without an explicit approval step.')


In [ ]:
# 6. Fully nested cross-validation — external outcome remains sealed
outer = StratifiedKFold(n_splits=OUTER_SPLITS,shuffle=True,random_state=SEED)
oof = pd.DataFrame({'outcome':y_dev})
fold_rows=[]; fold_params=[]

for model_i,(name,spec) in enumerate(MODEL_SPECS.items()):
    pred=np.full(len(dev),np.nan)
    for fold,(tr,va) in enumerate(outer.split(dev,y_dev),1):
        inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=SEED+100*model_i+fold)
        base=clone(spec['pipeline'])
        if spec['grid']:
            fit=GridSearchCV(base,spec['grid'],scoring='roc_auc',cv=inner,n_jobs=-1,refit=True,error_score=np.nan)
        else:
            fit=base
        fs=spec['features']
        fit.fit(dev.iloc[tr][fs],y_dev[tr])
        pred[va]=fit.predict_proba(dev.iloc[va][fs])[:,1]
        best=fit.best_params_ if hasattr(fit,'best_params_') else {}
        row={'model':name,'fold':fold,**metric_row(y_dev[va],pred[va],0.5)}
        fold_rows.append(row)
        fold_params.append({'model':name,'fold':fold,'best_params':json.dumps(best,default=str)})
        print(f'{name}: outer fold {fold}/{OUTER_SPLITS} complete')
    assert not np.isnan(pred).any()
    oof[name]=pred

oof.to_csv(ARTIFACTS/'all_models_nested_oof_predictions.csv',index=False)
pd.DataFrame(fold_rows).to_csv(TABLES/'S2_nested_fold_metrics.csv',index=False)
pd.DataFrame(fold_params).to_csv(ARTIFACTS/'nested_best_parameters.csv',index=False)

summary=[]
thresholds={}
for name in MODEL_SPECS:
    p=oof[name].to_numpy(); thresholds[name]=threshold_for_sensitivity(y_dev,p)
    row={'model':name,**metric_row(y_dev,p,thresholds[name])}
    lo,hi=bootstrap_ci(y_dev,p,roc_auc_score,n_boot=1000,seed=SEED+len(summary))
    row.update({'AUROC_CI_low':lo,'AUROC_CI_high':hi})
    summary.append(row)
internal_summary=pd.DataFrame(summary).sort_values(['AUROC','Brier'],ascending=[False,True])
internal_summary.to_csv(TABLES/'Table2_internal_model_performance.csv',index=False)

# Champion is chosen only from nested OOF performance. Ties favor calibration and simplicity.
champion=internal_summary.iloc[0]['model']
print('Internally selected champion:',champion)
display(internal_summary)


In [ ]:
# 7. Internal model-comparison figure, styled after the example paper
def plot_performance_panel(y,preds,title,path_png,path_pdf):
    fig,axes=plt.subplots(2,2,figsize=(12,10))
    ax_roc,ax_cal,ax_pr,ax_dca=axes.ravel()
    thresholds_grid=np.linspace(.05,.50,100)
    prevalence=y.mean()
    for i,(name,p) in enumerate(preds.items()):
        color=PALETTE[i%len(PALETTE)]
        fpr,tpr,_=roc_curve(y,p)
        ax_roc.plot(fpr,tpr,lw=1.7,color=color,label=f'{name} ({roc_auc_score(y,p):.3f})')
        bins=pd.qcut(p,q=min(10,len(np.unique(p))),duplicates='drop')
        cal=pd.DataFrame({'y':y,'p':p,'bin':bins}).groupby('bin',observed=True).agg(observed=('y','mean'),predicted=('p','mean'))
        ax_cal.plot(cal.predicted,cal.observed,'o-',lw=1.4,color=color,label=name)
        precision,recall,_=precision_recall_curve(y,p)
        ax_pr.plot(recall,precision,lw=1.7,color=color,label=f'{name} ({average_precision_score(y,p):.3f})')
        ax_dca.plot(thresholds_grid,net_benefit(y,p,thresholds_grid),lw=1.5,color=color,label=name)
    ax_roc.plot([0,1],[0,1],'--',color='gray'); ax_roc.set(xlabel='1 - Specificity',ylabel='Sensitivity',title='A. ROC curves')
    ax_cal.plot([0,1],[0,1],'--',color='gray'); ax_cal.set(xlabel='Predicted probability',ylabel='Observed proportion',title='B. Calibration')
    ax_pr.axhline(prevalence,ls='--',color='gray'); ax_pr.set(xlabel='Recall',ylabel='Precision',title='C. Precision-recall curves')
    treat_all=prevalence-(1-prevalence)*(thresholds_grid/(1-thresholds_grid))
    ax_dca.plot(thresholds_grid,treat_all,'--',color='black',label='Screen all'); ax_dca.axhline(0,ls=':',color='gray',label='Screen none')
    # Show the prespecified operating point inside the DCA panel. The locked
    # threshold is selected from development OOF predictions and is never
    # re-estimated on the external cohort.
    if champion in preds and champion in thresholds:
        locked_t_plot=float(thresholds[champion])
        locked_nb_plot=net_benefit(y,preds[champion],[locked_t_plot])[0]
        ax_dca.axvline(locked_t_plot,ls='-.',lw=1.3,color='#D55E00',
                       label=f'Locked threshold ({locked_t_plot:.3f})')
        ax_dca.scatter([locked_t_plot],[locked_nb_plot],s=38,color='#D55E00',zorder=5)
    ax_dca.set(xlabel='Threshold probability',ylabel='Net benefit',
               title='D. Decision curve analysis',xlim=(.05,.50))
    for ax in axes.ravel(): ax.legend(frameon=False,fontsize=7)
    fig.suptitle(title,fontsize=14); fig.tight_layout()
    fig.savefig(path_png,dpi=300,bbox_inches='tight'); fig.savefig(path_pdf,bbox_inches='tight'); plt.show()

internal_preds={name:oof[name].to_numpy() for name in MODEL_SPECS}
plot_performance_panel(y_dev,internal_preds,
    'Figure 2. Nested out-of-fold model performance — development cohort',
    FIG_MAIN/'Figure2_internal_performance.png',FIG_MAIN/'Figure2_internal_performance.pdf')

# Supplementary baseline-versus-champion comparison, mirroring the sample paper's S3 layout.
plot_performance_panel(y_dev,{'DemographicLR':internal_preds['DemographicLR'],champion:internal_preds[champion]},
    'Supplementary Figure S2. Small baseline versus selected model',
    FIG_SUPP/'FigureS2_baseline_vs_champion.png',FIG_SUPP/'FigureS2_baseline_vs_champion.pdf')


## 4. Stage 3 — final lock and external validation

Each model is tuned once more using development data only and refitted on all 2,000 development participants. A JSON lock is written before external MCI labels are accessed. The exact preprocessing objects and screening thresholds are then applied unchanged to the external cohort.


In [ ]:
# 8. Final development refit, pre-external lock, and external validation
import joblib

final_models={}; final_params={}
full_inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=SEED+9000)
for name,spec in MODEL_SPECS.items():
    base=clone(spec['pipeline']); fs=spec['features']
    if spec['grid']:
        fit=GridSearchCV(base,spec['grid'],scoring='roc_auc',cv=full_inner,n_jobs=-1,refit=True,error_score=np.nan)
    else: fit=base
    fit.fit(dev[fs],y_dev)
    final_models[name]=fit
    final_params[name]=fit.best_params_ if hasattr(fit,'best_params_') else {}
    print(name,'final development fit complete')

analysis_lock={
    'status':'LOCKED_BEFORE_EXTERNAL_OUTCOME_ACCESS',
    'target':'6CIT-defined screen-positive cognitive impairment (6CIT ≥ 8)',
    'development_n':len(dev),'events':int(y_dev.sum()),
    'candidate_predictors':43,'final_predictors':FEATURES_36,
    'dropped_missing_gt25':DROP_MISSING_GT25,'dropped_high_correlation':DROP_HIGH_CORR,
    'outer_cv':5,'inner_cv':3,'champion':champion,
    'model_hyperparameters':final_params,
    'threshold_rule':'highest nested-OOF threshold achieving sensitivity >= 0.80',
    'screening_thresholds':thresholds,
    'external_outcome_accessed_at_lock':False,
    'recalibration_in_primary_external_analysis':False,
    'seed':SEED,
}
with open(ARTIFACTS/'ANALYSIS_LOCK_BEFORE_EXTERNAL.json','w',encoding='utf-8') as f:
    json.dump(analysis_lock,f,indent=2,default=str)
print('Analysis lock written. External outcome may now be accessed.')

y_ext=clean_external_outcome(ext_raw)
assert len(y_ext)==1165 and int(y_ext.sum())==614
external_predictions={'outcome':y_ext}
external_rows=[]
for name,spec in MODEL_SPECS.items():
    p=final_models[name].predict_proba(ext_X[spec['features']])[:,1]
    external_predictions[name]=p
    row={'model':name,**metric_row(y_ext,p,thresholds[name])}
    lo,hi=bootstrap_ci(y_ext,p,roc_auc_score,n_boot=1000,seed=SEED+100+len(external_rows))
    row.update({'AUROC_CI_low':lo,'AUROC_CI_high':hi})
    external_rows.append(row)

external_predictions=pd.DataFrame(external_predictions)
external_summary=pd.DataFrame(external_rows).sort_values(['AUROC','Brier'],ascending=[False,True])
external_predictions.to_csv(ARTIFACTS/'external_locked_predictions.csv',index=False)
external_summary.to_csv(TABLES/'Table3_external_model_performance.csv',index=False)
joblib.dump(final_models[champion],ARTIFACTS/'champion_model.joblib')
display(external_summary)

external_preds={name:external_predictions[name].to_numpy() for name in MODEL_SPECS}
plot_performance_panel(y_ext,external_preds,
    'Figure 3. Locked external validation',
    FIG_MAIN/'Figure3_external_performance.png',FIG_MAIN/'Figure3_external_performance.pdf')


<!-- STUDY_FLOW_EXCLUDED -->
The study-flow figure is intentionally not generated in this run.

## 5. Stage 4 — interpretability, subgroup performance and education sensitivity

SHAP is applied only after model and external validation are locked. SHAP values describe model attribution, not causality. Subgroup performance uses the same frozen model; no subgroup-specific refitting is performed. Education sensitivity repeats the selected model family after removing education, while preserving the fixed 6CIT cutoff.


In [ ]:
# 10. Cross-model permutation importance and feature-ranking agreement
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

importance={}
for name,spec in MODEL_SPECS.items():
    result=permutation_importance(final_models[name],ext_X[spec['features']],y_ext,
        scoring='roc_auc',n_repeats=10,random_state=SEED,n_jobs=1)
    s=pd.Series(result.importances_mean,index=spec['features'])
    s=s.reindex(FEATURES_36).fillna(0)
    importance[name]=s/(s.abs().sum()+1e-12)
    print(name,'permutation importance complete')
importance=pd.DataFrame(importance)
importance.to_csv(TABLES/'cross_model_normalized_permutation_importance.csv')
rank_corr=importance.rank(ascending=False).corr(method='spearman')
rank_corr.to_csv(TABLES/'cross_model_feature_rank_correlations.csv')

fig,axes=plt.subplots(1,2,figsize=(16,8),gridspec_kw={'width_ratios':[1.6,1]})
sns.heatmap(importance,center=0,cmap='vlag',ax=axes[0],cbar_kws={'label':'Normalized AUROC decrease'})
axes[0].set_title('A. Normalized feature importance across models')
sns.heatmap(rank_corr,annot=True,fmt='.2f',vmin=-1,vmax=1,center=0,cmap='vlag',ax=axes[1],cbar_kws={'label':'Spearman rho'})
axes[1].set_title('B. Correlation of feature rankings')
fig.suptitle('Figure 5. Cross-model feature utilization',fontsize=14)
fig.tight_layout(); fig.savefig(FIG_MAIN/'Figure5_feature_importance_agreement.png',dpi=300,bbox_inches='tight'); fig.savefig(FIG_MAIN/'Figure5_feature_importance_agreement.pdf',bbox_inches='tight'); plt.show()


In [ ]:
# 11. SHAP analysis for the internally selected champion
import shap

champion_fit=final_models[champion]
champion_pipe=champion_fit.best_estimator_ if hasattr(champion_fit,'best_estimator_') else champion_fit
champion_features=MODEL_SPECS[champion]['features']
sample=ext_X[champion_features].sample(min(250,len(ext_X)),random_state=SEED)
background=dev[champion_features].sample(min(100,len(dev)),random_state=SEED)

prep=champion_pipe.named_steps['prep']; model=champion_pipe.named_steps['model']
X_bg=prep.transform(background); X_shap=prep.transform(sample)
try:
    transformed_names=np.asarray(prep.get_feature_names_out(),dtype=str)
except Exception:
    transformed_names=np.asarray([f'x{i}' for i in range(X_shap.shape[1])])

tree_names={'RandomForestClassifier','GradientBoostingClassifier','XGBClassifier','CatBoostClassifier','LGBMClassifier'}
if model.__class__.__name__ in tree_names:
    explainer=shap.TreeExplainer(model)
    explanation=explainer(X_shap)
elif model.__class__.__name__=='LogisticRegression':
    explainer=shap.LinearExplainer(model,X_bg)
    explanation=explainer(X_shap)
else:
    # Model-agnostic fallback is intentionally sampled for tractability.
    f=lambda z:model.predict_proba(z)[:,1]
    explainer=shap.Explainer(f,X_bg,algorithm='permutation')
    explanation=explainer(X_shap,max_evals=max(2*X_shap.shape[1]+1,101))

if getattr(explanation.values,'ndim',2)==3:
    explanation=explanation[:,:,1]
explanation.feature_names=transformed_names.tolist()

plt.figure(); shap.plots.beeswarm(explanation,max_display=20,show=False); plt.title(f'SHAP summary — {champion} (external cohort)'); plt.tight_layout(); plt.savefig(FIG_MAIN/'Figure6_SHAP_beeswarm.png',dpi=300,bbox_inches='tight'); plt.savefig(FIG_MAIN/'Figure6_SHAP_beeswarm.pdf',bbox_inches='tight'); plt.show()
plt.figure(); shap.plots.bar(explanation,max_display=20,show=False); plt.title(f'Mean absolute SHAP — {champion}'); plt.tight_layout(); plt.savefig(FIG_SUPP/'FigureS3_SHAP_bar.png',dpi=300,bbox_inches='tight'); plt.savefig(FIG_SUPP/'FigureS3_SHAP_bar.pdf',bbox_inches='tight'); plt.show()

mean_abs=pd.Series(np.abs(explanation.values).mean(axis=0),index=transformed_names).sort_values(ascending=False)
mean_abs.to_csv(TABLES/'SHAP_mean_absolute_values.csv',header=['mean_abs_SHAP'])
print('Top SHAP features:'); display(mean_abs.head(20).to_frame('mean_abs_SHAP'))


In [ ]:
# 12. Frozen-model subgroup analysis: sex, age, and education
def subgroup_metrics(frame,y,p,variable,groups):
    rows=[]
    for label,mask in groups.items():
        idx=np.asarray(mask)
        yy=y[idx]; pp=p[idx]
        if len(yy)<20 or len(np.unique(yy))<2:
            rows.append({'variable':variable,'group':label,'n':len(yy),'events':int(yy.sum()),'AUROC':np.nan})
            continue
        row={'variable':variable,'group':label,'n':len(yy),'events':int(yy.sum()),**metric_row(yy,pp,thresholds[champion])}
        lo,hi=bootstrap_ci(yy,pp,roc_auc_score,n_boot=1000,seed=SEED+len(rows)+300)
        row.update({'AUROC_CI_low':lo,'AUROC_CI_high':hi}); rows.append(row)
    return rows

p_ext=external_predictions[champion].to_numpy()
sub_rows=[]
sex=ext_X['sex'].astype(str).str.lower()
sub_rows+=subgroup_metrics(ext_X,y_ext,p_ext,'sex',{'female':sex.eq('female'),'male':sex.eq('male')})
sub_rows+=subgroup_metrics(ext_X,y_ext,p_ext,'age',{'<70':ext_X['age']<70,'>=70':ext_X['age']>=70})
edu=ext_X['EDU']
sub_rows+=subgroup_metrics(ext_X,y_ext,p_ext,'education',{
    'illiterate_or_primary':edu.eq('illiterate&primary'),
    'secondary_or_academic':edu.isin(['secondary','academic'])})
subgroups=pd.DataFrame(sub_rows)
subgroups.to_csv(TABLES/'S5_external_subgroup_performance.csv',index=False)
display(subgroups)

plot=subgroups.dropna(subset=['AUROC']).copy()
fig,ax=plt.subplots(figsize=(8,5.5))
ypos=np.arange(len(plot)); ax.errorbar(plot.AUROC,ypos,xerr=[plot.AUROC-plot.AUROC_CI_low,plot.AUROC_CI_high-plot.AUROC],fmt='o',capsize=4,color=PALETTE[0])
ax.set_yticks(ypos,[f'{v}: {g} (n={n})' for v,g,n in zip(plot.variable,plot.group,plot.n)])
ax.axvline(roc_auc_score(y_ext,p_ext),ls='--',color='gray',label='Overall external AUROC')
ax.set(xlabel='AUROC (95% bootstrap CI)',title='Supplementary Figure S4. External subgroup performance',xlim=(0.4,1.0)); ax.legend(frameon=False); fig.tight_layout()
fig.savefig(FIG_SUPP/'FigureS4_subgroup_AUROC.png',dpi=300,bbox_inches='tight'); fig.savefig(FIG_SUPP/'FigureS4_subgroup_AUROC.pdf',bbox_inches='tight'); plt.show()


In [ ]:
# 13. Education sensitivity: selected model family refitted without EDU
noedu_features=[f for f in champion_features if f!='EDU']

def rebuild_champion_without_education():
    base=clone(MODEL_SPECS[champion]['pipeline'])
    # Rebuild preprocessing because the feature schema changed.
    ordinal=(champion=='TabPFN')
    demographic=(champion=='DemographicLR')
    prep=make_preprocessor(noedu_features,ordinal=ordinal,demographic=demographic)
    base.set_params(prep=prep)
    grid={k:v for k,v in MODEL_SPECS[champion]['grid'].items()}
    return base,grid

noedu_oof=np.full(len(dev),np.nan)
for fold,(tr,va) in enumerate(StratifiedKFold(5,shuffle=True,random_state=SEED+500).split(dev,y_dev),1):
    base,grid=rebuild_champion_without_education()
    if grid:
        fit=GridSearchCV(base,grid,scoring='roc_auc',cv=StratifiedKFold(3,shuffle=True,random_state=SEED+500+fold),n_jobs=-1,refit=True)
    else: fit=base
    fit.fit(dev.iloc[tr][noedu_features],y_dev[tr])
    noedu_oof[va]=fit.predict_proba(dev.iloc[va][noedu_features])[:,1]
    print('Education sensitivity fold',fold,'complete')

thr_noedu=threshold_for_sensitivity(y_dev,noedu_oof)
base,grid=rebuild_champion_without_education()
if grid:
    noedu_final=GridSearchCV(base,grid,scoring='roc_auc',cv=StratifiedKFold(3,shuffle=True,random_state=SEED+600),n_jobs=-1,refit=True).fit(dev[noedu_features],y_dev)
else: noedu_final=base.fit(dev[noedu_features],y_dev)
noedu_ext=noedu_final.predict_proba(ext_X[noedu_features])[:,1]

edu_sensitivity=pd.DataFrame([
    {'analysis':'With education — nested OOF',**metric_row(y_dev,oof[champion].to_numpy(),thresholds[champion])},
    {'analysis':'Without education — nested OOF',**metric_row(y_dev,noedu_oof,thr_noedu)},
    {'analysis':'With education — external',**metric_row(y_ext,p_ext,thresholds[champion])},
    {'analysis':'Without education — external',**metric_row(y_ext,noedu_ext,thr_noedu)},
])
edu_sensitivity.to_csv(TABLES/'S6_education_sensitivity.csv',index=False)
display(edu_sensitivity)

plot_performance_panel(y_ext,{'With education':p_ext,'Without education':noedu_ext},
    'Supplementary Figure S5. Education sensitivity — external cohort',
    FIG_SUPP/'FigureS5_education_sensitivity.png',FIG_SUPP/'FigureS5_education_sensitivity.pdf')


In [ ]:
# 14. Publication Table 1: cohort characteristics and transportability
def _fmt_continuous(s):
    x=pd.to_numeric(s,errors='coerce').dropna()
    if len(x)==0: return 'NA'
    q1,med,q3=np.quantile(x,[.25,.50,.75])
    return f'{med:.2f} [{q1:.2f}, {q3:.2f}]'

def _continuous_smd(a,b):
    a=pd.to_numeric(a,errors='coerce').dropna(); b=pd.to_numeric(b,errors='coerce').dropna()
    pooled=np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2)
    return float((a.mean()-b.mean())/pooled) if pooled>0 else np.nan

def _binary_smd(p1,p2):
    den=np.sqrt((p1*(1-p1)+p2*(1-p2))/2)
    return float((p1-p2)/den) if den>0 else np.nan

table1=[]
for level,value in [('Non-MCI',0),('MCI',1)]:
    p1=(y_dev==value).mean(); p2=(y_ext==value).mean()
    table1.append({'variable':'6CIT-defined outcome','level_or_statistic':level,
        'development':f'{(y_dev==value).sum()} ({100*p1:.1f}%)',
        'external':f'{(y_ext==value).sum()} ({100*p2:.1f}%)','SMD':_binary_smd(p1,p2)})

for feature in NUMERIC_36:
    a=dev[feature]; b=ext_X[feature]
    table1.append({'variable':feature,'level_or_statistic':'Median [IQR]',
        'development':_fmt_continuous(a),'external':_fmt_continuous(b),'SMD':_continuous_smd(a,b)})
    table1.append({'variable':feature,'level_or_statistic':'Missing, n (%)',
        'development':f'{a.isna().sum()} ({100*a.isna().mean():.1f}%)',
        'external':f'{b.isna().sum()} ({100*b.isna().mean():.1f}%)','SMD':np.nan})

for feature in CATEGORICAL_36:
    a=dev[feature]; b=ext_X[feature]
    levels=sorted(set(a.dropna().astype(str)).union(set(b.dropna().astype(str))))
    for level in levels:
        p1=a.astype(str).eq(level).mean(); p2=b.astype(str).eq(level).mean()
        table1.append({'variable':feature,'level_or_statistic':level,
            'development':f'{a.astype(str).eq(level).sum()} ({100*p1:.1f}%)',
            'external':f'{b.astype(str).eq(level).sum()} ({100*p2:.1f}%)','SMD':_binary_smd(p1,p2)})
    table1.append({'variable':feature,'level_or_statistic':'Missing, n (%)',
        'development':f'{a.isna().sum()} ({100*a.isna().mean():.1f}%)',
        'external':f'{b.isna().sum()} ({100*b.isna().mean():.1f}%)','SMD':np.nan})

table1=pd.DataFrame(table1)
table1['absolute_SMD']=table1['SMD'].abs()
table1.to_csv(TABLES/'Table1_cohort_characteristics.csv',index=False)

# Supplementary baseline table stratified by the fixed outcome definition.
def _group_summary(series,mask,numeric):
    s=series[mask]
    return _fmt_continuous(s) if numeric else None

by_outcome=[]
for feature in NUMERIC_36:
    row={'variable':feature,'level_or_statistic':'Median [IQR]'}
    for cohort,frame,y in [('development',dev,y_dev),('external',ext_X,y_ext)]:
        for label,value in [('non_MCI',0),('MCI',1)]:
            row[f'{cohort}_{label}']=_fmt_continuous(frame.loc[y==value,feature])
    by_outcome.append(row)
for feature in CATEGORICAL_36:
    levels=sorted(set(dev[feature].dropna().astype(str)).union(set(ext_X[feature].dropna().astype(str))))
    for level in levels:
        row={'variable':feature,'level_or_statistic':level}
        for cohort,frame,y in [('development',dev,y_dev),('external',ext_X,y_ext)]:
            for label,value in [('non_MCI',0),('MCI',1)]:
                s=frame.loc[y==value,feature]; n=len(s); count=s.astype(str).eq(level).sum()
                row[f'{cohort}_{label}']=f'{count} ({100*count/n:.1f}%)'
        by_outcome.append(row)
pd.DataFrame(by_outcome).to_csv(TABLES/'S1_baseline_characteristics_by_outcome.csv',index=False)
print('Publication Table 1 complete (SMDs reported; no baseline-significance p-values).')
display(table1.head(20))


In [ ]:
# 15. Complete 95% bootstrap confidence intervals for all performance metrics
REPORT_METRICS=['AUROC','AUPRC','Brier','calibration_intercept','calibration_slope',
                'accuracy','sensitivity','specificity','PPV','NPV','F1','MCC']

def _stratified_bootstrap_indices(y,rng):
    neg=np.flatnonzero(y==0); pos=np.flatnonzero(y==1)
    idx=np.concatenate([rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)])
    rng.shuffle(idx)
    return idx

def bootstrap_all_metric_cis(y,p,threshold,n_boot=1000,seed=SEED):
    y=np.asarray(y); p=np.asarray(p); point=metric_row(y,p,threshold)
    rng=np.random.default_rng(seed); draws={m:[] for m in REPORT_METRICS}
    for _ in range(n_boot):
        idx=_stratified_bootstrap_indices(y,rng)
        boot=metric_row(y[idx],p[idx],threshold)
        for metric in REPORT_METRICS: draws[metric].append(boot[metric])
    rows=[]
    for metric in REPORT_METRICS:
        lo,hi=np.quantile(draws[metric],[.025,.975])
        rows.append({'metric':metric,'estimate':point[metric],'CI_low':lo,'CI_high':hi})
    return rows

ci_rows=[]
cohort_specs=[('Nested OOF',y_dev,oof),('External locked',y_ext,external_predictions)]
for cohort,y,pred_frame in cohort_specs:
    for model_i,name in enumerate(MODEL_SPECS):
        for row in bootstrap_all_metric_cis(y,pred_frame[name].to_numpy(),thresholds[name],
                                             n_boot=1000,seed=SEED+1000*model_i+(0 if cohort=='Nested OOF' else 500)):
            ci_rows.append({'cohort':cohort,'model':name,'threshold':thresholds[name],**row})
performance_ci_long=pd.DataFrame(ci_rows)
performance_ci_long.to_csv(TABLES/'Table2_3_complete_performance_95CI_long.csv',index=False)
performance_ci_long['estimate_95CI']=performance_ci_long.apply(
    lambda r:f"{r.estimate:.3f} ({r.CI_low:.3f}–{r.CI_high:.3f})",axis=1)
performance_ci_wide=performance_ci_long.pivot(index=['cohort','model','threshold'],columns='metric',values='estimate_95CI').reset_index()
performance_ci_wide.to_csv(TABLES/'Table2_3_complete_performance_95CI_formatted.csv',index=False)
print('Complete 95% CIs written for discrimination, calibration and threshold metrics.')
display(performance_ci_wide)


In [ ]:
# 16. Paired model comparisons: DeLong AUROC tests and paired-bootstrap effect sizes
from scipy.stats import norm

def _compute_midrank(x):
    order=np.argsort(x); z=x[order]; n=len(x); ranks=np.zeros(n,float); i=0
    while i<n:
        j=i
        while j<n and z[j]==z[i]: j+=1
        ranks[i:j]=0.5*(i+j-1)
        i=j
    out=np.empty(n,float); out[order]=ranks+1
    return out

def _fast_delong(predictions_sorted_transposed,label_1_count):
    m=label_1_count; n=predictions_sorted_transposed.shape[1]-m; k=predictions_sorted_transposed.shape[0]
    pos=predictions_sorted_transposed[:,:m]; neg=predictions_sorted_transposed[:,m:]
    tx=np.empty((k,m)); ty=np.empty((k,n)); tz=np.empty((k,m+n))
    for r in range(k):
        tx[r]=_compute_midrank(pos[r]); ty[r]=_compute_midrank(neg[r]); tz[r]=_compute_midrank(predictions_sorted_transposed[r])
    aucs=tz[:,:m].sum(axis=1)/m/n-(m+1)/(2*n)
    v01=(tz[:,:m]-tx)/n; v10=1-(tz[:,m:]-ty)/m
    cov=np.cov(v01)/m+np.cov(v10)/n
    return aucs,cov

def delong_paired_test(y,p1,p2):
    y=np.asarray(y,dtype=int); order=np.argsort(-y)
    aucs,cov=_fast_delong(np.vstack([p1,p2])[:,order],int(y.sum()))
    contrast=np.array([1.,-1.]); var=float(contrast@cov@contrast)
    if not np.isfinite(var) or var<=0: return np.nan
    return float(2*norm.sf(abs(aucs[0]-aucs[1])/np.sqrt(var)))

def _paired_bootstrap_delta(y,p1,p2,metric,n_boot=2000,seed=SEED):
    y=np.asarray(y); p1=np.asarray(p1); p2=np.asarray(p2); rng=np.random.default_rng(seed); vals=[]
    for _ in range(n_boot):
        idx=_stratified_bootstrap_indices(y,rng)
        vals.append(metric(y[idx],p1[idx])-metric(y[idx],p2[idx]))
    return np.quantile(vals,[.025,.975])

def _bh_fdr(pvalues):
    p=np.asarray(pvalues,float); out=np.full(len(p),np.nan); valid=np.flatnonzero(np.isfinite(p))
    if len(valid)==0: return out
    order=valid[np.argsort(p[valid])]; ranked=p[order]*len(order)/np.arange(1,len(order)+1)
    ranked=np.minimum.accumulate(ranked[::-1])[::-1]
    out[order]=np.minimum(ranked,1.0); return out

comparison_rows=[]
for cohort,y,pred_frame in cohort_specs:
    p_champ=pred_frame[champion].to_numpy()
    for j,other in enumerate(MODEL_SPECS):
        if other==champion: continue
        p_other=pred_frame[other].to_numpy()
        lo,hi=_paired_bootstrap_delta(y,p_champ,p_other,roc_auc_score,n_boot=2000,
                                      seed=SEED+700+j+(0 if cohort=='Nested OOF' else 100))
        comparison_rows.append({'cohort':cohort,'champion':champion,'comparator':other,
            'champion_AUROC':roc_auc_score(y,p_champ),'comparator_AUROC':roc_auc_score(y,p_other),
            'delta_AUROC':roc_auc_score(y,p_champ)-roc_auc_score(y,p_other),
            'delta_AUROC_CI_low':lo,'delta_AUROC_CI_high':hi,
            'DeLong_p':delong_paired_test(y,p_champ,p_other),
            'primary_comparison':other=='DemographicLR'})
paired_comparisons=pd.DataFrame(comparison_rows)
paired_comparisons['DeLong_p_FDR']=np.nan
for cohort in paired_comparisons.cohort.unique():
    mask=paired_comparisons.cohort.eq(cohort)
    paired_comparisons.loc[mask,'DeLong_p_FDR']=_bh_fdr(paired_comparisons.loc[mask,'DeLong_p'])
paired_comparisons.to_csv(TABLES/'Table4_paired_AUROC_comparisons.csv',index=False)
print('Positive delta favors the preselected champion; external results do not alter model selection.')
display(paired_comparisons)


In [ ]:
# 17. Numeric DCA summary, calibration-in-the-large, and clinical-usability verdict
def _logit_scalar(x):
    x=float(np.clip(x,1e-6,1-1e-6)); return np.log(x/(1-x))

calibration_rows=[]
for cohort,y,pred_frame in cohort_specs:
    for name in MODEL_SPECS:
        p=pred_frame[name].to_numpy(); ci,cs=calibration_intercept_slope(y,p)
        calibration_rows.append({'cohort':cohort,'model':name,
            'observed_prevalence':y.mean(),'mean_predicted_risk':p.mean(),
            'calibration_in_the_large':_logit_scalar(y.mean())-_logit_scalar(p.mean()),
            'calibration_intercept_joint_model':ci,'calibration_slope':cs})
calibration_details=pd.DataFrame(calibration_rows)
calibration_details.to_csv(TABLES/'Table3_calibration_details.csv',index=False)

# Extend the numeric DCA grid through 0.50 and include the exact locked
# operating threshold (0.463606 in the current analysis), not only rounded
# 0.01 increments.
dca_grid=np.unique(np.r_[np.round(np.arange(.05,.501,.01),2),
                         round(float(thresholds[champion]),6)])
dca_rows=[]
for t in dca_grid:
    all_nb=y_ext.mean()-(1-y_ext.mean())*t/(1-t)
    for name in [champion,'DemographicLR']:
        model_nb=net_benefit(y_ext,external_predictions[name].to_numpy(),[t])[0]
        dca_rows.append({'threshold':t,'model':name,'net_benefit':model_nb,
                         'screen_all_net_benefit':all_nb,'screen_none_net_benefit':0.0,
                         'increment_vs_best_default':model_nb-max(all_nb,0.0)})
dca_numeric=pd.DataFrame(dca_rows)
dca_numeric.to_csv(TABLES/'Table5_external_DCA_numeric.csv',index=False)

champ_dca=dca_numeric[dca_numeric.model.eq(champion)]
beneficial=champ_dca.loc[champ_dca.increment_vs_best_default>0,'threshold'].to_numpy()
locked_t=thresholds[champion]
locked_nb=net_benefit(y_ext,p_ext,[locked_t])[0]
locked_all=y_ext.mean()-(1-y_ext.mean())*locked_t/(1-locked_t)
dca_summary={'model':champion,'evaluated_threshold_range':'0.05–0.50',
    'beneficial_threshold_min':float(beneficial.min()) if len(beneficial) else None,
    'beneficial_threshold_max':float(beneficial.max()) if len(beneficial) else None,
    'proportion_of_grid_beneficial':float(len(beneficial)/len(dca_grid)),
    'locked_screening_threshold':float(locked_t),'net_benefit_at_locked_threshold':float(locked_nb),
    'increment_vs_best_default_at_locked_threshold':float(locked_nb-max(locked_all,0.0))}
with open(ARTIFACTS/'external_DCA_summary.json','w',encoding='utf-8') as f: json.dump(dca_summary,f,indent=2)

champ_ext=metric_row(y_ext,p_ext,locked_t)
if champ_ext['AUROC']>=.80 and not (.90<=champ_ext['calibration_slope']<=1.10):
    verdict='CONDITIONALLY USABLE — good discrimination but external recalibration/prospective validation is required.'
elif champ_ext['AUROC']<.70 or len(beneficial)==0:
    verdict='NOT CLINICALLY USABLE under the prespecified criteria.'
else:
    verdict='CONDITIONALLY USABLE pending prospective validation and clinician oversight.'
clinical_summary={'model':champion,'external_metrics':champ_ext,'DCA':dca_summary,
                  'verdict':verdict,'deployment_recommendation':'Do not deploy without prospective validation and clinician oversight.'}
with open(ROOT/'clinical_usability_summary.json','w',encoding='utf-8') as f: json.dump(clinical_summary,f,indent=2,default=float)
print(verdict); display(calibration_details); display(pd.DataFrame([dca_summary]))


In [ ]:
# 18. Formal bootstrap contrasts for sex, age, and education subgroups
subgroup_definitions={
    'sex':{'female':sex.eq('female'),'male':sex.eq('male')},
    'age':{'<70':ext_X['age']<70,'>=70':ext_X['age']>=70},
    'education':{'illiterate_or_primary':edu.eq('illiterate&primary'),
                 'secondary_or_academic':edu.isin(['secondary','academic'])},
}

def subgroup_auc_contrast(y,p,mask1,mask2,n_boot=2000,seed=SEED):
    y=np.asarray(y); p=np.asarray(p); i1=np.flatnonzero(np.asarray(mask1)); i2=np.flatnonzero(np.asarray(mask2))
    point=roc_auc_score(y[i1],p[i1])-roc_auc_score(y[i2],p[i2])
    rng=np.random.default_rng(seed); vals=[]
    for _ in range(n_boot):
        a=i1[_stratified_bootstrap_indices(y[i1],rng)]; b=i2[_stratified_bootstrap_indices(y[i2],rng)]
        vals.append(roc_auc_score(y[a],p[a])-roc_auc_score(y[b],p[b]))
    vals=np.asarray(vals); lo,hi=np.quantile(vals,[.025,.975])
    p_boot=min(1.0,2*min((vals<=0).mean(),(vals>=0).mean()))
    return point,lo,hi,p_boot

contrast_rows=[]
for j,(variable,groups) in enumerate(subgroup_definitions.items()):
    (g1,m1),(g2,m2)=list(groups.items())
    delta,lo,hi,pval=subgroup_auc_contrast(y_ext,p_ext,m1,m2,n_boot=2000,seed=SEED+900+j)
    contrast_rows.append({'variable':variable,'group_1':g1,'group_2':g2,
                          'delta_AUROC_group1_minus_group2':delta,'CI_low':lo,'CI_high':hi,
                          'bootstrap_p_two_sided':pval,
                          'n_group_1':int(np.asarray(m1).sum()),'n_group_2':int(np.asarray(m2).sum())})
subgroup_contrasts=pd.DataFrame(contrast_rows)
subgroup_contrasts['p_FDR']=_bh_fdr(subgroup_contrasts['bootstrap_p_two_sided'])
subgroup_contrasts.to_csv(TABLES/'S5b_external_subgroup_AUROC_contrasts.csv',index=False)
display(subgroup_contrasts)


In [ ]:
# 19. Complete SHAP dependence, local case, and interactive force-plot outputs
import re

top5=mean_abs.head(5).index.tolist()
dependence_manifest=[]
for rank,feature in enumerate(top5,1):
    safe=re.sub(r'[^A-Za-z0-9_]+','_',feature)
    plt.figure(figsize=(7,5))
    shap.dependence_plot(feature,explanation.values,X_shap,feature_names=transformed_names.tolist(),
                         interaction_index='auto',show=False)
    plt.title(f'SHAP dependence — {feature} ({champion}, external sample n={len(sample)})')
    plt.tight_layout()
    png=FIG_SUPP/f'FigureS6_{rank}_SHAP_dependence_{safe}.png'
    pdf=FIG_SUPP/f'FigureS6_{rank}_SHAP_dependence_{safe}.pdf'
    plt.savefig(png,dpi=300,bbox_inches='tight'); plt.savefig(pdf,bbox_inches='tight'); plt.show()
    dependence_manifest.append({'rank':rank,'feature':feature,'png':png.name,'pdf':pdf.name})
pd.DataFrame(dependence_manifest).to_csv(TABLES/'SHAP_dependence_plot_manifest.csv',index=False)

sample_rows=sample.index.to_numpy(dtype=int)
sample_prob=p_ext[sample_rows]; sample_y=y_ext[sample_rows]
sample_class=(sample_prob>=thresholds[champion]).astype(int)
case_masks={'TP':(sample_y==1)&(sample_class==1),'FP':(sample_y==0)&(sample_class==1),
            'FN':(sample_y==1)&(sample_class==0),'TN':(sample_y==0)&(sample_class==0)}
case_manifest=[]; selected_positions={}
for case,mask in case_masks.items():
    positions=np.flatnonzero(mask)
    if len(positions)==0: continue
    median_prob=np.median(sample_prob[positions])
    pos=int(positions[np.argmin(np.abs(sample_prob[positions]-median_prob))])
    selected_positions[case]=pos
    plt.figure(figsize=(9,6)); shap.plots.waterfall(explanation[pos],max_display=15,show=False)
    plt.title(f'{case} representative case — predicted risk {sample_prob[pos]:.3f}')
    plt.tight_layout()
    plt.savefig(FIG_SUPP/f'FigureS7_SHAP_waterfall_{case}.png',dpi=300,bbox_inches='tight')
    plt.savefig(FIG_SUPP/f'FigureS7_SHAP_waterfall_{case}.pdf',bbox_inches='tight'); plt.show()
    case_manifest.append({'case_type':case,'anonymous_row_index':int(sample_rows[pos]),
                          'outcome':int(sample_y[pos]),'predicted_probability':float(sample_prob[pos]),
                          'threshold':float(thresholds[champion])})
pd.DataFrame(case_manifest).to_csv(TABLES/'SHAP_local_case_manifest.csv',index=False)

force_pos=selected_positions.get('TP',next(iter(selected_positions.values())))
force_plot=shap.force_plot(float(np.asarray(explanation.base_values[force_pos]).reshape(-1)[0]),
                           explanation.values[force_pos],X_shap[force_pos],
                           feature_names=transformed_names.tolist(),matplotlib=False)
shap.save_html(str(FIG_SUPP/'FigureS8_SHAP_force_representative_case.html'),force_plot)

# Human-readable category-to-ordinal mapping for interpreting TabPFN SHAP directions.
mapping=[]
try:
    cat_encoder=prep.named_transformers_['cat'].named_steps['encode']
    for feature,categories in zip(CATEGORICAL_36,cat_encoder.categories_[:len(CATEGORICAL_36)]):
        for code,category in enumerate(categories): mapping.append({'feature':feature,'encoded_value':code,'category':category})
except Exception as exc:
    mapping.append({'feature':'mapping_unavailable','encoded_value':np.nan,'category':str(exc)})
pd.DataFrame(mapping).to_csv(TABLES/'S8_SHAP_categorical_value_mapping.csv',index=False)
print('SHAP dependence plots, TP/FP/FN/TN waterfalls, and interactive force plot complete.')


In [ ]:
# 20. Publication-output manifest
manifest={
    'main_figures':{
        'Figure 1':'Data quality, outcome distribution, missingness and correlation',
        'Figure 2':'Nested OOF ROC, calibration, PR and DCA for all models',
        'Figure 3':'Locked external ROC, calibration, PR and DCA for all models',
        'Figure 4':'Participant flow and analysis design',
        'Figure 5':'Cross-model normalized importance and rank correlation',
        'Figure 6':'SHAP beeswarm for the selected model',
    },
    'supplementary_figures':{
        'S1':'Feature availability across cohorts',
        'S2':'Small demographic baseline versus selected model',
        'S3':'SHAP bar plot',
        'S4':'External subgroup performance by sex, age, and education',
        'S5':'Education sensitivity with fixed 6CIT outcome definition',
        'S6':'SHAP dependence plots for the five leading features',
        'S7':'SHAP waterfalls for representative TP, FP, FN, and TN cases',
        'S8':'Interactive SHAP force plot',
    },
    'tables':{
        'S2':'Per-fold nested-CV metrics','S5':'Subgroup performance',
        'S6':'Education sensitivity','S7':'Feature availability and exclusions',
        'Table 1':'Cohort characteristics and standardized mean differences',
        'Table 2':'Internal model performance with complete 95% CIs',
        'Table 3':'External model performance and calibration with complete 95% CIs',
        'Table 4':'Paired DeLong AUROC comparisons with FDR correction',
        'Table 5':'Numeric decision-curve analysis',
        'S5b':'Formal subgroup AUROC contrasts','S8':'SHAP category-code mapping',
    },
    'reporting_standards':['TRIPOD+AI','PROBAST'],
    'clinical_interpretation':'Associational prediction only; SHAP is not causal; prospective validation required before deployment.'
}
with open(ROOT/'publication_output_manifest.json','w',encoding='utf-8') as f:json.dump(manifest,f,indent=2)
all_files=sorted(str(p.relative_to(ROOT)) for p in ROOT.rglob('*') if p.is_file())
pd.DataFrame({'file':all_files}).to_csv(ROOT/'output_index.csv',index=False)
print(f'Pipeline complete. {len(all_files)} files written under {ROOT}')
display(pd.DataFrame({'file':all_files}))


<!-- SUBMISSION_FIGURE_BUNDLE_V1 -->
## Final manuscript figure bundle
This cell regenerates the five analytical figures in the main manuscript and four supplementary figures from the locked analysis outputs. The study-flow figure is excluded.

In [ ]:
# SUBMISSION_FIGURE_BUNDLE_V1
# Reproduce the nine manuscript and supplementary analytical figures.
# The study-flow figure is intentionally excluded.
from pathlib import Path
_submission_figure_source = '"""Generate all figures embedded in the final manuscript and supplement.\n\nThis script is executed as the final cell of the Moshabeh Kaggle notebook. It\nuses only objects and locked outputs created earlier in that notebook. It does\nnot refit, retune, or recalibrate any prediction model.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport shutil\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import LinearSegmentedColormap, LogNorm\nfrom matplotlib.lines import Line2D\nfrom matplotlib.ticker import FormatStrFormatter\nfrom PIL import Image, ImageDraw, ImageFont\nimport shap\nimport statsmodels.api as sm\nfrom patsy import dmatrix, build_design_matrices\nfrom sklearn.metrics import (\n    average_precision_score,\n    precision_recall_curve,\n    roc_auc_score,\n    roc_curve,\n)\n\n\nROOT = Path(globals()["ROOT"])\nFIG_MAIN = Path(globals()["FIG_MAIN"])\nFIG_SUPP = Path(globals()["FIG_SUPP"])\nTABLES = Path(globals()["TABLES"])\nOUT = ROOT / "submission_figure_bundle"\nMAIN_OUT = OUT / "main_figures"\nSUPP_OUT = OUT / "supplementary_figures"\nif OUT.exists():\n    shutil.rmtree(OUT)\nMAIN_OUT.mkdir(parents=True, exist_ok=True)\nSUPP_OUT.mkdir(parents=True, exist_ok=True)\n\nchampion = globals()["champion"]\ny_dev = np.asarray(globals()["y_dev"], dtype=int)\ny_ext = np.asarray(globals()["y_ext"], dtype=int)\noof = globals()["oof"]\nexternal_predictions = globals()["external_predictions"]\nthresholds = globals()["thresholds"]\nnet_benefit = globals()["net_benefit"]\nmissingness = globals()["missingness"]\nimportance = globals()["importance"].copy()\nrank_corr = globals()["rank_corr"].copy()\nexplanation = globals()["explanation"]\nX_shap = np.asarray(globals()["X_shap"])\ntransformed_names = np.asarray(globals()["transformed_names"], dtype=str)\nsample = globals()["sample"]\n\n\nBLUE = "#2C78B8"\nORANGE = "#D55E00"\nINK = "#26313C"\nGREY = "#555555"\nPALETTE_LOCAL = list(plt.get_cmap("tab10").colors)\n\n\nDISPLAY = {\n    "cat__EDU": "Education",\n    "num__age": "Age",\n    "cat__sleep_quality": "Sleep quality",\n    "cat__iadl": "IADL",\n    "num__visual_acuity_both": "Binocular visual acuity",\n    "cat__household_income": "Household income",\n    "cat__missingindicator_iadl": "IADL missing (indicator)",\n    "num__waist_hip_ratio": "Waist-to-hip ratio",\n    "cat__adl": "ADL",\n    "cat__employment_status": "Employment status",\n    "cat__depression_status": "Depression status",\n    "num__whisper_test_left": "Whisper test, left ear",\n    "num__whisper_test_right": "Whisper test, right ear",\n    "num__Albumin": "Albumin",\n    "num__BMI": "Body-mass index",\n    "cat__osteoporosis_status": "Osteoporosis",\n    "cat__hypertension": "Hypertension",\n    "num__max_handgrip": "Maximum handgrip strength",\n    "cat__sex": "Sex",\n    "EDU": "Education",\n    "iadl": "IADL",\n    "visual_acuity_both": "Binocular visual acuity",\n    "household_income": "Household income",\n    "sleep_quality": "Sleep quality",\n    "employment_status": "Employment status",\n    "max_handgrip": "Maximum handgrip strength",\n    "depression_status": "Depression status",\n    "adl": "ADL",\n    "BMI": "BMI",\n    "osteoporosis_status": "Osteoporosis status",\n    "age": "Age",\n}\n\n\ndef display_name(raw: str) -> str:\n    raw = str(raw).strip()\n    return DISPLAY.get(\n        raw,\n        raw.replace("cat__", "").replace("num__", "").replace("_", " ").capitalize(),\n    )\n\n\ndef save_figure(fig, folder: Path, stem: str, dpi: int = 600) -> None:\n    fig.savefig(folder / f"{stem}.png", dpi=dpi, bbox_inches="tight", facecolor="white")\n    fig.savefig(folder / f"{stem}.pdf", bbox_inches="tight", facecolor="white")\n    plt.close(fig)\n\n\ndef save_pil(image: Image.Image, folder: Path, stem: str, dpi: int = 300) -> None:\n    image.save(folder / f"{stem}.png", dpi=(dpi, dpi), optimize=True)\n    image.save(folder / f"{stem}.pdf", "PDF", resolution=dpi)\n\n\ndef as_probability_array(frame, name: str) -> np.ndarray:\n    value = frame[name]\n    return value.to_numpy(dtype=float) if hasattr(value, "to_numpy") else np.asarray(value, dtype=float)\n\n\n# ---------------------------------------------------------------------------\n# Figures 2 and S2. ROC, calibration, precision-recall, and DCA\n# ---------------------------------------------------------------------------\ndef performance_panel(y, predictions, stem, folder, suptitle=None):\n    fig, axes = plt.subplots(2, 2, figsize=(12, 10))\n    ax_roc, ax_cal, ax_pr, ax_dca = axes.ravel()\n    grid = np.linspace(0.05, 0.50, 100)\n    prevalence = float(np.mean(y))\n    for i, (name, p) in enumerate(predictions.items()):\n        p = np.asarray(p, dtype=float)\n        color = PALETTE_LOCAL[i % len(PALETTE_LOCAL)]\n        fpr, tpr, _ = roc_curve(y, p)\n        ax_roc.plot(fpr, tpr, lw=1.7, color=color,\n                    label=f"{name} ({roc_auc_score(y, p):.3f})")\n        bins = pd.qcut(p, q=min(10, len(np.unique(p))), duplicates="drop")\n        cal = (pd.DataFrame({"y": y, "p": p, "bin": bins})\n               .groupby("bin", observed=True)\n               .agg(observed=("y", "mean"), predicted=("p", "mean")))\n        ax_cal.plot(cal.predicted, cal.observed, "o-", lw=1.4, color=color, label=name)\n        precision, recall, _ = precision_recall_curve(y, p)\n        ax_pr.plot(recall, precision, lw=1.7, color=color,\n                   label=f"{name} ({average_precision_score(y, p):.3f})")\n        ax_dca.plot(grid, net_benefit(y, p, grid), lw=1.5, color=color, label=name)\n    ax_roc.plot([0, 1], [0, 1], "--", color="gray")\n    ax_roc.set(xlabel="1 - Specificity", ylabel="Sensitivity", title="A. ROC curves")\n    ax_cal.plot([0, 1], [0, 1], "--", color="gray")\n    ax_cal.set(xlabel="Predicted probability", ylabel="Observed proportion", title="B. Calibration")\n    ax_pr.axhline(prevalence, ls="--", color="gray")\n    ax_pr.set(xlabel="Recall", ylabel="Precision", title="C. Precision-recall curves")\n    treat_all = prevalence - (1 - prevalence) * grid / (1 - grid)\n    ax_dca.plot(grid, treat_all, "--", color="black", label="Screen all")\n    ax_dca.axhline(0, ls=":", color="gray", label="Screen none")\n    if champion in predictions:\n        locked = float(thresholds[champion])\n        nb = float(net_benefit(y, np.asarray(predictions[champion]), [locked])[0])\n        ax_dca.axvline(locked, ls="-.", lw=1.3, color=ORANGE,\n                       label=f"Locked threshold ({locked:.3f})")\n        ax_dca.scatter([locked], [nb], s=38, color=ORANGE, zorder=5)\n    ax_dca.set(xlabel="Threshold probability", ylabel="Net benefit",\n               title="D. Decision curve analysis", xlim=(0.05, 0.50))\n    for ax in axes.ravel():\n        ax.legend(frameon=False, fontsize=7)\n    if suptitle:\n        fig.suptitle(suptitle, fontsize=14)\n    fig.tight_layout(rect=[0, 0, 1, 0.97] if suptitle else None)\n    save_figure(fig, folder, stem, dpi=600)\n\n\ndef figure2_and_s2():\n    model_order = [name for name in globals()["MODEL_SPECS"]]\n    ext_preds = {name: as_probability_array(external_predictions, name) for name in model_order}\n    performance_panel(y_ext, ext_preds, "Figure2_External_validation_performance", MAIN_OUT)\n    dev_preds = {\n        "DemographicLR": as_probability_array(oof, "DemographicLR"),\n        champion: as_probability_array(oof, champion),\n    }\n    performance_panel(\n        y_dev, dev_preds, "Supplementary_Figure_S2_Development_baseline_vs_selected_model",\n        SUPP_OUT, "Supplementary Figure S2. Small baseline versus selected model",\n    )\n\n\n# ---------------------------------------------------------------------------\n# Figure 3. Participant-level agreement of predicted probabilities\n# ---------------------------------------------------------------------------\ndef figure3_hexbin():\n    models = [("DemographicLR", "Demographic baseline"),\n              ("ElasticNet", "Elastic-net logistic regression")]\n    cmap = LinearSegmentedColormap.from_list(\n        "count_blue", ["#DCE8F3", "#92BAD8", "#397DAD", "#174F7A", "#092D49"]\n    )\n    tab = as_probability_array(external_predictions, champion)\n    fig = plt.figure(figsize=(190 / 25.4, 98 / 25.4))\n    gs = fig.add_gridspec(1, 3, left=0.085, right=0.925, bottom=0.235, top=0.825,\n                          width_ratios=[1, 1, 0.045], wspace=0.23)\n    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]\n    cax = fig.add_subplot(gs[0, 2])\n    collections = []\n    for i, (ax, (key, title)) in enumerate(zip(axes, models)):\n        comp = as_probability_array(external_predictions, key)\n        coll = ax.hexbin(comp, tab, gridsize=(28, 16), extent=(0, 1, 0, 1),\n                         mincnt=1, cmap=cmap, bins="log", linewidths=0.16,\n                         edgecolors="face", zorder=2)\n        collections.append(coll)\n        ax.plot([0, 1], [0, 1], color="#333333", linewidth=0.85,\n                linestyle=(0, (4, 3)), zorder=3)\n        ax.set(xlim=(0, 1), ylim=(0, 1), aspect="equal",\n               xlabel="Comparator predicted probability")\n        ax.set_xticks(np.linspace(0, 1, 6)); ax.set_yticks(np.linspace(0, 1, 6))\n        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))\n        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))\n        ax.set_title(title, loc="left", pad=8)\n        ax.text(-0.015, 1.105, chr(65 + i), transform=ax.transAxes,\n                fontweight="bold", fontsize=11)\n        ax.spines[["top", "right"]].set_visible(False); ax.grid(False)\n    axes[0].set_ylabel("TabPFN predicted probability")\n    cb = fig.colorbar(collections[0], cax=cax)\n    cb.set_label("Participants per hexagon (log scale)", fontsize=7.5, labelpad=8)\n    fig.legend(handles=[Line2D([0], [0], color="#333333", linewidth=0.85,\n                               linestyle=(0, (4, 3)), label="Equal predicted probabilities")],\n               loc="lower center", bbox_to_anchor=(0.47, 0.055), frameon=False,\n               handlelength=2.6, fontsize=7.5)\n    save_figure(fig, MAIN_OUT, "Figure3_Prediction_agreement_hexbin", dpi=600)\n\n\n# ---------------------------------------------------------------------------\n# Figure 4. Flexible calibration\n# ---------------------------------------------------------------------------\ndef _logit(p):\n    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)\n    return np.log(p / (1 - p))\n\n\ndef flexible_calibration(y, p, cohort, spline_df=4):\n    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)\n    lp = _logit(p)\n    design = dmatrix(f"cr(lp, df={spline_df})", {"lp": lp}, return_type="dataframe")\n    model = sm.GLM(y, design, family=sm.families.Binomial()).fit()\n    grid = np.linspace(p.min(), p.max(), 300)\n    grid_design = build_design_matrices([design.design_info], {"lp": _logit(grid)})[0]\n    sf = model.get_prediction(grid_design).summary_frame(alpha=0.05)\n    fitted = np.asarray(model.predict(design))\n    error = np.abs(fitted - p)\n    bins = pd.qcut(p, q=min(10, len(np.unique(p))), duplicates="drop")\n    dec = (pd.DataFrame({"y": y, "p": p, "bin": bins})\n           .groupby("bin", observed=True)\n           .agg(predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size"))\n           .reset_index(drop=True))\n    return grid, sf, dec, {"ICI": float(error.mean()), "E90": float(np.quantile(error, 0.90))}\n\n\ndef figure4_calibration():\n    analyses = [\n        ("Nested OOF", y_dev, as_probability_array(oof, champion)),\n        ("External locked", y_ext, as_probability_array(external_predictions, champion)),\n    ]\n    palette = {"Nested OOF": "#0072B2", "External locked": ORANGE}\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), sharex=True, sharey=True)\n    for ax, (name, y, p) in zip(axes, analyses):\n        grid, sf, dec, summary = flexible_calibration(y, p, name)\n        color = palette[name]\n        ax.fill_between(grid, sf["mean_ci_lower"].to_numpy(), sf["mean_ci_upper"].to_numpy(),\n                        color=color, alpha=0.18, linewidth=0, label="Pointwise 95% CI")\n        ax.plot(grid, sf["mean"].to_numpy(), color=color, linewidth=2.2,\n                label="Flexible spline curve")\n        ax.scatter(dec["predicted"], dec["observed"],\n                   s=np.clip(dec["n"] * 0.35, 35, 95), facecolor="white",\n                   edgecolor=color, linewidth=1.4, zorder=4, label="Risk deciles")\n        ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1.1,\n                label="Ideal calibration")\n        inset = ax.inset_axes([0.08, 0.04, 0.86, 0.15])\n        inset.hist(p, bins=25, color=color, alpha=0.28)\n        inset.set_xlim(0, 1); inset.set_xticks([]); inset.set_yticks([])\n        for spine in inset.spines.values(): spine.set_visible(False)\n        inset.set_facecolor("none")\n        ax.set_title(f"{name}\\nICI={summary[\'ICI\']:.3f}; E90={summary[\'E90\']:.3f}")\n        ax.set_xlabel("Predicted probability of screen-positive cognitive impairment")\n        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal", adjustable="box")\n    axes[0].set_ylabel("Observed probability of screen-positive cognitive impairment")\n    handles, labels = axes[1].get_legend_handles_labels()\n    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.01),\n               ncol=4, frameon=False)\n    fig.tight_layout(rect=[0, 0.08, 1, 1])\n    save_figure(fig, MAIN_OUT, "Figure4_Flexible_calibration", dpi=600)\n\n\n# ---------------------------------------------------------------------------\n# Figure 5. SHAP, subgroup discrimination, and education sensitivity\n# ---------------------------------------------------------------------------\ndef fig_to_image(fig, dpi=300):\n    from io import BytesIO\n    buf = BytesIO()\n    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", facecolor="white")\n    plt.close(fig); buf.seek(0)\n    return Image.open(buf).convert("RGB")\n\n\ndef fit_image(img, box_w, box_h):\n    scale = min(box_w / img.width, box_h / img.height)\n    return img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))), Image.LANCZOS)\n\n\nFONT_DIR = Path(matplotlib.__file__).parent / "mpl-data" / "fonts" / "ttf"\nF_REG = str(FONT_DIR / "DejaVuSans.ttf")\nF_BOLD = str(FONT_DIR / "DejaVuSans-Bold.ttf")\n\n\ndef pil_font(size, bold=False):\n    return ImageFont.truetype(F_BOLD if bold else F_REG, size)\n\n\ndef paste_panel(canvas, img, box, letter, title, letter_size=58, title_size=42):\n    x, y, w, h = box\n    draw = ImageDraw.Draw(canvas)\n    head = max(letter_size, title_size) + 24\n    draw.text((x, y), letter, font=pil_font(letter_size, True), fill="black")\n    draw.text((x + letter_size + 16, y + 8), title, font=pil_font(title_size), fill="black")\n    fitted = fit_image(img, w, h - head)\n    canvas.paste(fitted, (x + (w - fitted.width) // 2, y + head))\n\n\ndef panel_shap_beeswarm():\n    values = np.asarray(explanation.values)\n    top_idx = np.argsort(np.mean(np.abs(values), axis=0))[::-1][:19]\n    exp_top = explanation[:, top_idx]\n    exp_top.feature_names = [display_name(transformed_names[i]) for i in top_idx]\n    plt.figure(figsize=(8.2, 10.5))\n    shap.plots.beeswarm(exp_top, max_display=19, show=False, plot_size=None)\n    plt.xlabel("SHAP value (impact on model output)")\n    plt.title("")\n    return fig_to_image(plt.gcf(), dpi=300)\n\n\ndef panel_subgroups():\n    rows = pd.read_csv(TABLES / "S5_external_subgroup_performance.csv")\n    pretty = {\n        ("education", "illiterate_or_primary"): "Illiterate / primary",\n        ("education", "secondary_or_academic"): "Secondary / academic",\n        ("age", "<70"): "Age < 70 y", ("age", ">=70"): "Age ≥ 70 y",\n        ("sex", "female"): "Women", ("sex", "male"): "Men",\n    }\n    order = list(pretty)\n    idx = {(r.variable, r.group): r for r in rows.itertuples()}\n    overall = roc_auc_score(y_ext, as_probability_array(external_predictions, champion))\n    fig, ax = plt.subplots(figsize=(7.6, 4.4))\n    ypos = list(range(len(order)))[::-1]\n    for y, key in zip(ypos, order):\n        r = idx[key]; est, lo, hi = float(r.AUROC), float(r.AUROC_CI_low), float(r.AUROC_CI_high)\n        color = ORANGE if key[0] == "education" else BLUE\n        ax.plot([lo, hi], [y, y], color=color, lw=2.4, solid_capstyle="round")\n        ax.plot([est], [y], "o", color=color, ms=8)\n        ax.text(1.005, y, f"{est:.3f} ({lo:.3f}–{hi:.3f})", va="center",\n                fontsize=9.5, transform=ax.get_yaxis_transform())\n    ax.axvline(overall, color=GREY, ls="--", lw=1.5,\n               label=f"Overall external AUROC ({overall:.3f})")\n    ax.set_yticks(ypos, [f"{pretty[k]}  (n={int(idx[k].n)})" for k in order], fontsize=10.5)\n    ax.set_xlim(0.55, 1.0); ax.set_xlabel("AUROC (95% bootstrap CI)")\n    ax.grid(axis="x", alpha=0.35); ax.set_axisbelow(True)\n    ax.spines[["top", "right", "left"]].set_visible(False)\n    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), frameon=False)\n    return fig_to_image(fig)\n\n\ndef panel_education():\n    rows = pd.read_csv(TABLES / "S6_education_sensitivity.csv").set_index("analysis")\n    settings = [\n        ("Development\\n(nested OOF)", "With education — nested OOF", "Without education — nested OOF"),\n        ("External\\nvalidation", "With education — external", "Without education — external"),\n    ]\n    fig, ax = plt.subplots(figsize=(6.4, 4.4))\n    for x, (label, full_key, no_key) in enumerate(settings):\n        full = float(rows.loc[full_key, "AUROC"]); noedu = float(rows.loc[no_key, "AUROC"])\n        ax.plot([x - 0.13, x + 0.13], [full, noedu], color=GREY, lw=1.6)\n        ax.plot(x - 0.13, full, "o", color=BLUE, ms=11)\n        ax.plot(x + 0.13, noedu, "o", color=ORANGE, ms=11)\n        ax.annotate(f"{full:.3f}", (x - 0.13, full), xytext=(0, 12),\n                    textcoords="offset points", ha="center", color=BLUE)\n        ax.annotate(f"{noedu:.3f}", (x + 0.13, noedu), xytext=(0, -20),\n                    textcoords="offset points", ha="center", color=ORANGE)\n        ax.annotate(f"Δ = {noedu - full:+.3f}", (x, (full + noedu) / 2),\n                    xytext=(26, -3), textcoords="offset points", ha="left")\n    ax.set_xticks([0, 1], [x[0] for x in settings]); ax.set_ylabel("AUROC")\n    ax.set_xlim(-0.45, 1.55); ax.set_ylim(0.782, 0.883)\n    ax.grid(axis="y", alpha=0.35); ax.spines[["top", "right"]].set_visible(False)\n    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=BLUE, label="Full predictor set"),\n                       Line2D([], [], marker="o", ls="", color=ORANGE, label="Education removed")],\n              loc="upper left", frameon=False)\n    return fig_to_image(fig)\n\n\ndef figure5_shap_subgroups():\n    left = panel_shap_beeswarm(); top = panel_subgroups(); bottom = panel_education()\n    width, height, margin, gap = 4400, 2920, 110, 110\n    canvas = Image.new("RGB", (width, height), "white")\n    body_top = 60\n    left_w = int((width - 2 * margin - gap) * 0.46)\n    right_w = width - 2 * margin - gap - left_w\n    body_h = height - body_top - margin\n    paste_panel(canvas, left, (margin, body_top, left_w, body_h), "A",\n                "SHAP summary — TabPFN, external cohort")\n    rh = (body_h - gap) // 2\n    paste_panel(canvas, top, (margin + left_w + gap, body_top, right_w, rh), "B",\n                "External discrimination by subgroup")\n    paste_panel(canvas, bottom, (margin + left_w + gap, body_top + rh + gap, right_w, rh), "C",\n                "Sensitivity to removing education")\n    save_pil(canvas, MAIN_OUT, "Figure5_SHAP_subgroups_education", dpi=300)\n\n\n# ---------------------------------------------------------------------------\n# Figure 6. Cross-model importance and ranking agreement\n# ---------------------------------------------------------------------------\ndef figure6_importance():\n    models = ["ElasticNet", "SVM", "RF", "GBM", "XGBoost", "CatBoost", "LightGBM", "TabPFN"]\n    all_models = ["DemographicLR", *models]\n    ranks = importance[models].rank(ascending=False, method="average")\n    order = pd.DataFrame({"median_rank": ranks.median(axis=1), "mean_rank": ranks.mean(axis=1)})\n    order["feature"] = order.index\n    top = order.sort_values(["median_rank", "mean_rank", "feature"]).head(12).index.tolist()\n    subcorr = rank_corr.loc[models, models]\n    labels = DISPLAY\n    model_labels = {"DemographicLR": "Demographic baseline", "ElasticNet": "Elastic-net",\n                    **{m: m for m in models if m != "ElasticNet"}}\n    short = ["EN", "SVM", "RF", "GBM", "XGB", "CB", "LGBM", "TabPFN"]\n    rank_colors = ["#123D68", "#326FA4", "#86B3D6", "#D9E2EB"]\n    cmap = LinearSegmentedColormap.from_list(\n        "correlation_blue", ["#E9EFF5", "#87B4D6", "#286294", "#123D68"])\n    fig = plt.figure(figsize=(190 / 25.4, 190 / 25.4))\n    fig.text(0.04, 0.955, "A", fontsize=12, weight="bold")\n    ax = fig.add_axes([0.31, 0.505, 0.665, 0.420])\n    for row, feature in enumerate(top):\n        ax.axhline(row, color="#EDF0F3", linewidth=0.55, zorder=0)\n        for col, model in enumerate(models):\n            rank = float(ranks.loc[feature, model])\n            band = 0 if rank <= 3 else 1 if rank <= 6 else 2 if rank <= 12 else 3\n            ax.scatter(col, row, s=215, color=rank_colors[band], edgecolor="#CBD5DF", lw=0.25)\n            txt = str(int(rank)) if rank.is_integer() else f"{rank:.1f}"\n            ax.text(col, row, txt, ha="center", va="center", fontsize=7 if len(txt) < 4 else 6,\n                    color="white" if band <= 1 else "#24384B", weight="bold" if rank == 1 else "normal")\n    ax.set(xlim=(-0.5, 7.5), ylim=(11.55, -0.55))\n    ax.set_yticks(range(12), [labels.get(f, display_name(f)) for f in top], fontsize=8)\n    ax.set_xticks(range(8), [model_labels[m] for m in models], rotation=35, ha="right", fontsize=7.5)\n    ax.tick_params(axis="both", length=0, pad=8); ax.spines[:].set_visible(False)\n    for tick in ax.get_xticklabels():\n        if tick.get_text() == "TabPFN": tick.set_weight("bold")\n    legend = [Line2D([], [], ls="", marker="o", markersize=7, color=c, label=t)\n              for c, t in zip(rank_colors, ["Ranks 1–3", "4–6", "7–12", "13–36"])]\n    fig.legend(handles=legend, loc="center", bbox_to_anchor=(0.635, 0.405), ncol=4,\n               frameon=False, fontsize=7.5)\n    fig.text(0.04, 0.350, "B", fontsize=12, weight="bold")\n    bx = fig.add_axes([0.22, 0.040, 0.28, 0.260])\n    for i, model in enumerate(all_models):\n        value = float(importance.loc["EDU", model] * 100)\n        color = "#B65D16" if model == "TabPFN" else "#326FA4"\n        bx.hlines(i, 0, value, color=color, linewidth=1.6)\n        bx.scatter(value, i, color=color, s=28 if model == "TabPFN" else 22,\n                   marker="D" if model == "TabPFN" else "o")\n        bx.text(value + 3.5, i, f"{value:.1f}", va="center", fontsize=7.5,\n                weight="bold" if model == "TabPFN" else "normal")\n    bx.set(xlim=(0, 100), ylim=(8.55, -0.55), xlabel="Normalized importance (%)")\n    bx.set_xticks([0, 25, 50, 75, 100])\n    bx.set_yticks(range(9), [model_labels[m] for m in all_models], fontsize=7)\n    bx.tick_params(axis="y", length=0, pad=5); bx.spines[["top", "right", "left"]].set_visible(False)\n    for tick in bx.get_yticklabels():\n        if tick.get_text() == "TabPFN": tick.set_weight("bold")\n    fig.text(0.575, 0.350, "C", fontsize=12, weight="bold")\n    cx = fig.add_axes([0.645, 0.040, 0.315, 0.260])\n    for row in range(8):\n        for col in range(8):\n            rho = float(subcorr.iloc[row, col])\n            if row == col:\n                cx.text(col, row, "—", ha="center", va="center", fontsize=7, color="#909AA5")\n            else:\n                cx.scatter(col, row, s=230 * rho, color=cmap(rho), lw=0)\n                cx.text(col, row, f"{rho:.2f}".removeprefix("0"), ha="center", va="center",\n                        fontsize=6.1, color="white" if rho >= 0.56 else "#18354E")\n    cx.set(xlim=(-0.5, 7.5), ylim=(7.5, -0.5), aspect="equal")\n    cx.set_xticks(range(8), short, rotation=45, ha="right", fontsize=6.5)\n    cx.set_yticks(range(8), short, fontsize=6.5)\n    cx.tick_params(axis="both", length=0, pad=4); cx.spines[:].set_visible(False)\n    save_figure(fig, MAIN_OUT, "Figure6_Predictor_importance_and_rank_agreement", dpi=600)\n\n\n# ---------------------------------------------------------------------------\n# Supplementary Figures S1, S3 and S4\n# ---------------------------------------------------------------------------\ndef copy_existing(stem_source, stem_target):\n    for suffix in (".png", ".pdf"):\n        source = stem_source.with_suffix(suffix)\n        if not source.exists():\n            raise FileNotFoundError(source)\n        shutil.copy2(source, SUPP_OUT / f"{stem_target}{suffix}")\n\n\ndef supplementary_s1():\n    copy_existing(FIG_SUPP / "FigureS1_missingness", "Supplementary_Figure_S1_Missingness")\n\n\nAXIS_LABEL = {\n    "cat__EDU": "Education  (0 = academic, 1 = illiterate/primary, 2 = secondary)",\n    "num__age": "Age (standardized)",\n    "cat__sleep_quality": "Sleep quality  (0 = good, 1 = poor)",\n    "cat__iadl": "IADL  (0 = dependent, 1 = independent)",\n    "num__visual_acuity_both": "Binocular visual acuity (standardized)",\n}\n\n\ndef supplementary_s3_dependence():\n    top_idx = np.argsort(np.mean(np.abs(np.asarray(explanation.values)), axis=0))[::-1][:5]\n    pretty_names = [display_name(x) for x in transformed_names]\n    fig = plt.figure(figsize=(13.6, 14.3))\n    gs = fig.add_gridspec(3, 2, hspace=0.42, wspace=0.28)\n    for rank, idx in enumerate(top_idx):\n        ax = fig.add_subplot(gs[rank // 2, rank % 2])\n        shap.dependence_plot(\n            int(idx), np.asarray(explanation.values), X_shap,\n            feature_names=pretty_names, interaction_index="auto", ax=ax, show=False,\n        )\n        raw = transformed_names[idx]\n        ax.set_xlabel(AXIS_LABEL.get(raw, display_name(raw)))\n        ax.set_ylabel(f"SHAP value for {display_name(raw)}")\n        ax.set_title(f"{chr(65 + rank)}  {display_name(raw)}", loc="left", fontweight="bold")\n    empty = fig.add_subplot(gs[2, 1]); empty.axis("off")\n    save_figure(fig, SUPP_OUT, "Supplementary_Figure_S3_SHAP_dependence", dpi=300)\n\n\ndef crop_title(img, fraction=0.045):\n    out = img.convert("RGB").copy()\n    ImageDraw.Draw(out).rectangle([0, 0, out.width, int(out.height * fraction)], fill="white")\n    return out\n\n\ndef supplementary_s4_cases():\n    manifest = pd.read_csv(TABLES / "SHAP_local_case_manifest.csv").set_index("case_type")\n    order = ["TP", "TN", "FP", "FN"]\n    labels = {"TP": "True positive", "TN": "True negative",\n              "FP": "False positive", "FN": "False negative"}\n    width, height, margin, gap = 4290, 4730, 100, 90\n    cell_w = (width - 2 * margin - gap) // 2\n    cell_h = (height - 2 * margin - gap) // 2\n    canvas = Image.new("RGB", (width, height), "white")\n    for i, case in enumerate(order):\n        source = FIG_SUPP / f"FigureS7_SHAP_waterfall_{case}.png"\n        img = crop_title(Image.open(source), 0.040)\n        r, c = divmod(i, 2)\n        box = (margin + c * (cell_w + gap), margin + r * (cell_h + gap), cell_w, cell_h)\n        p = float(manifest.loc[case, "predicted_probability"])\n        paste_panel(canvas, img, box, chr(65 + i), f"{labels[case]} — predicted risk {p:.3f}",\n                    letter_size=52, title_size=40)\n    save_pil(canvas, SUPP_OUT, "Supplementary_Figure_S4_SHAP_local_cases", dpi=300)\n\n\ndef create_manifest_and_zip():\n    records = []\n    for group, folder in (("main", MAIN_OUT), ("supplementary", SUPP_OUT)):\n        for path in sorted(folder.glob("*")):\n            if path.suffix.lower() in {".png", ".pdf"}:\n                records.append({"group": group, "file": path.name, "bytes": path.stat().st_size})\n    pd.DataFrame(records).to_csv(OUT / "figure_manifest.csv", index=False)\n    summary = {\n        "main_figure_count": len(list(MAIN_OUT.glob("*.png"))),\n        "supplementary_figure_count": len(list(SUPP_OUT.glob("*.png"))),\n        "png_count": len(list(OUT.rglob("*.png"))),\n        "pdf_count": len(list(OUT.rglob("*.pdf"))),\n        "model_refits_in_this_script": 0,\n        "source": "locked outputs and in-memory objects from the completed Moshabeh notebook",\n    }\n    (OUT / "README.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    zip_path = ROOT / "Moshabeh_all_manuscript_figures.zip"\n    if zip_path.exists(): zip_path.unlink()\n    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for path in sorted(OUT.rglob("*")):\n            if path.is_file(): zf.write(path, path.relative_to(OUT.parent))\n        script_path = ROOT / "generate_all_submission_figures.py"\n        if script_path.exists(): zf.write(script_path, script_path.name)\n    print(json.dumps(summary, indent=2))\n    print("ZIP ready:", zip_path)\n\n\nfigure2_and_s2()\nfigure3_hexbin()\nfigure4_calibration()\nfigure5_shap_subgroups()\nfigure6_importance()\nsupplementary_s1()\nsupplementary_s3_dependence()\nsupplementary_s4_cases()\ncreate_manifest_and_zip()\n'
_submission_figure_script = ROOT / 'generate_all_submission_figures.py'
_submission_figure_script.write_text(_submission_figure_source, encoding='utf-8')
exec(compile(_submission_figure_source, str(_submission_figure_script), 'exec'), globals())
